[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/six_hundred_cell.ipynb)

# The 600-cell from the icosahedron's mirrors

The three mirrors of the icosahedron, multiplied in pairs, give three rotors. Multiplied with one another in every way, the rotors close up into a group of 120 unit spinors, two for each of the 60 rotations of the icosahedron, one the negative of the other. A unit spinor of three-dimensional space is four numbers of unit length, a point of the three-sphere, and these 120 points are the vertices of a regular polytope in four dimensions: the 600-cell, each vertex with twelve nearest neighbours, the polytope's 600 cells tetrahedra.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.math.reflections import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Even = ga.gatype.even()
# The angles at which the mirrors meet: the first and the second, the first and the third, the second and
# the third.
FIRST_SECOND = np.pi / 2
FIRST_THIRD = np.pi / 3
SECOND_THIRD = np.pi / 5
# The longest products of turns taken: enough to reach every element of the group.
LENGTH = 9
FRAMES = 90
FRAME_MS = 50

## 1. The mirrors and their turns

The mirrors are those of the icosahedron: the first is $x$, the second $x$ turned in the $xy$ plane, the third its part in their plane, from the reciprocal vectors of the first two, plus a part along the plane's normal, with the pairings $-\cos(\pi / m)$ between them. The turns are their products in pairs.

In [ ]:
first = mv.x                                                                    # [] Vector
second = ((mv.x ^ mv.y) * (-(np.pi - FIRST_SECOND) / 2)).exp() >> first        # [] Vector
plane = first ^ second                                                          # [] Bivector
in_plane = -np.cos(FIRST_THIRD) * (second | plane.inverse()) - np.cos(SECOND_THIRD) * (plane.inverse() | first)   # [] Vector
normal = plane.normalized().dual()                                             # [] Vector
# Along the normal goes what the part in the plane leaves of a unit length.
height = (1 - in_plane.squared()).square_root()                                 # [] Scalar
third = in_plane + height * normal                                              # [] Vector
# The turns about the axes where two mirrors meet, named by how many of them make a scalar.
turns = stack([second * first, third * first, third * second])                 # [twofold, threefold, fivefold] Rotor

## 2. Every product of the turns

Starting from one, each round multiplies everything so far by one, or by each of the three turns. After a few rounds the products repeat each other: they have closed up into the group. Many of the products are the same spinor, reached along different routes; there are 120 different ones.

In [ ]:
steps = concatenate([mv.even([[1.0, 0.0, 0.0, 0.0]]), turns])                   # [steps] Even
spinors = steps                                                                 # [products] Even
for _ in range(LENGTH - 1):
    spinors = (spinors[:, None] * steps[None, :]).reshape((-1,))                # [products] Even

## 3. The 600-cell

The spinors are drawn as points of the three-sphere, in perspective into three dimensions, with an edge between each spinor and its twelve nearest neighbours. The polytope turns in four dimensions: multiplying every spinor by a unit spinor on the left and another on the right turns the three-sphere in two planes at once, and the group of spinors into itself, moved.

In [ ]:
def turning(frames: int):
    """The spinors turned on the left in the xy plane and on the right in the yz plane, the right turn
    at a fifth of the rate, per frame."""
    for angle in np.linspace(0.0, 2 * np.pi, frames, endpoint=False):
        yield (mv.xy * (angle / 2)).exp() * spinors * (mv.yz * (angle / 10)).exp()


display(Image(filename=save_animation(render.animate_polytope(turning(FRAMES)), "six_hundred_cell", FRAME_MS)))

In [ ]:
# checks
# The products are unit spinors, 120 different ones; the product of any two of them is one of them;
# each has twelve nearest neighbours, at the edge length of the 600-cell, one over the golden ratio.
vertices = render.distinct(spinors.kernel)                                      # [120, 4]
assert len(vertices) == 120
np.testing.assert_allclose(np.linalg.norm(vertices, axis=-1), 1.0, atol=1e-12)
group = mv.even(vertices)                                                       # [120] Even
products = render.distinct((group[:, None] * group[None, :]).reshape((-1,)).kernel)
assert len(products) == 120 and np.abs(products[:, None] - vertices[None]).max(axis=-1).min(axis=-1).max() < 1e-9
gaps = np.linalg.norm(vertices[:, None] - vertices[None], axis=-1)
golden = (1 + np.sqrt(5)) / 2
np.testing.assert_array_equal((np.abs(gaps - 1 / golden) < 1e-9).sum(axis=-1), 12)